Generate the Trapi Query Graph from Natural Language Input using OntoGPT
 
 This agent uses OntoGPT to convert natural language input into a TRAPI 1.1 compliant query graph.
 
 ## Features
 
 - Converts natural language queries into structured TRAPI query graphs.
 - Utilizes OntoGPT templates for specific biomedical queries.
 - Integrates with the Biolink Model for accurate entity and relationship representation.

 ## Configure OAK / OntoGPT API Key

OntoGPT does not take a model name or API key directly in your Python code.  
Instead, it reads credentials from the OAK configuration system.  
Set your API key once on the machine running the agent:  

For OpenAI (gpt-4o):
```bash
   runoak set-apikey -e openai YOUR_OPENAI_KEY
```
For Google Gemini:
```bash
   runoak set-apikey -e google YOUR_GEMINI_KEY
```
This will create/update:
```bash
   ~/.config/oak/apikeys.yml
```
OntoGPT will automatically use this configuration whenever it is called.

In [14]:
from query_graph_builder_agent import NLP2TRAPIAgent

agent = NLP2TRAPIAgent()

result = agent.process_question("Which genes are associated with Parkinson disease?")
print(result)

{'question': 'Which genes are associated with Parkinson disease?', 'disease_curie': 'MONDO:0005180', 'trapi_query': {'message': {'query_graph': {'nodes': {'n0': {'ids': ['MONDO:0005180'], 'category': 'biolink:Disease'}, 'n1': {'category': 'biolink:ChemicalSubstance'}}, 'edges': {'e0': {'subject': 'n1', 'object': 'n0', 'predicate': 'biolink:treats'}}}}}, 'ontogpt_output': {'input_text': 'Which genes are associated with Parkinson disease?', 'raw_completion_output': 'diseases: Parkinson disease\ndrug: \ndrug_to_disease_relationships: ', 'prompt': 'Split the following piece of text into fields in the following format:\n\ndiseases: <A semicolon-separated list of diseases mentioned in the text.>\ndrug: <A semicolon-separated list of drugs mentioned in the text.>\ndrug_to_disease_relationships: <A semicolon-separated list of relationships, where each is a triple connecting a drug to a disease. Each relationship should have a type (predicate) and two entities (subject and object). The predicat

In [13]:
import json
print(json.dumps(result['trapi_query'], indent=2))

{
  "message": {
    "query_graph": {
      "nodes": {
        "n0": {
          "ids": [
            "MONDO:0005027"
          ],
          "category": "biolink:Disease"
        },
        "n1": {
          "category": "biolink:ChemicalSubstance"
        }
      },
      "edges": {
        "e0": {
          "subject": "n1",
          "object": "n0",
          "predicate": "biolink:treats"
        }
      }
    }
  }
}


In [ ]:
# use trapi api to return results
import httpx

trapi_query = result["trapi_query"]

ROBOTRAPI_URL = (
    "https://robokop-automat.apps.renci.org/monarch-kg/query"
    "?profile=false&validate=true&subclass=true"
)

response = await httpx.AsyncClient().post(ROBOTRAPI_URL, json=trapi_query, timeout=60)
trapi_response = response.json()

In [6]:
print(json.dumps(trapi_response, indent=2))

{
  "message": {
    "query_graph": {
      "nodes": {
        "n0": {
          "ids": [
            "MONDO:0005027"
          ],
          "categories": null,
          "set_interpretation": "BATCH",
          "constraints": [],
          "member_ids": [],
          "category": "biolink:Disease"
        },
        "n1": {
          "ids": null,
          "categories": null,
          "set_interpretation": "BATCH",
          "constraints": [],
          "member_ids": [],
          "category": "biolink:ChemicalSubstance"
        }
      },
      "edges": {
        "e0": {
          "subject": "n1",
          "object": "n0",
          "knowledge_type": null,
          "predicates": null,
          "attribute_constraints": [],
          "qualifier_constraints": [],
          "predicate": "biolink:treats"
        }
      }
    },
    "knowledge_graph": {
      "nodes": {
        "MONDO:0005027": {
          "name": "epilepsy",
          "categories": [
            "biolink:BiologicalEntit

In [4]:
import requests
import os

In [5]:
MONARCH_TRAPI_URL = os.getenv(
    "MONARCH_TRAPI_URL",
    "https://robokop-automat.apps.renci.org/monarch-kg/query",
)

In [6]:
subclass = True
validate = True    
params = {}
if subclass:
        params["subclass"] = "true"
if validate:
        params["validate"] = "true"

In [7]:
params

{'subclass': 'true', 'validate': 'true'}

In [8]:
payload = {
  "message": {
    "query_graph": {
      "edges": {
        "e0": {
          "object": "n0",
          "predicate": "biolink:treats",
          "subject": "n1"
        }
      },
      "nodes": {
        "n0": {
          "category": "biolink:Disease",
          "ids": [
            "MONDO:0005148"
          ]
        },
        "n1": {
          "category": "biolink:ChemicalSubstance"
        }
      }
    }
  }
}


In [9]:
resp = requests.post(MONARCH_TRAPI_URL, params=params, json=payload, timeout=60)

In [11]:
resp.json()

{'message': {'query_graph': {'nodes': {'n0': {'ids': ['MONDO:0005148'],
     'categories': None,
     'set_interpretation': 'BATCH',
     'constraints': [],
     'member_ids': [],
     'category': 'biolink:Disease'},
    'n1': {'ids': None,
     'categories': None,
     'set_interpretation': 'BATCH',
     'constraints': [],
     'member_ids': [],
     'category': 'biolink:ChemicalSubstance'}},
   'edges': {'e0': {'subject': 'n1',
     'object': 'n0',
     'knowledge_type': None,
     'predicates': None,
     'attribute_constraints': [],
     'qualifier_constraints': [],
     'predicate': 'biolink:treats'}}},
  'knowledge_graph': {'nodes': {'MONDO:0005148': {'name': 'type 2 diabetes mellitus',
     'categories': ['biolink:BiologicalEntity',
      'biolink:Disease',
      'biolink:DiseaseOrPhenotypicFeature',
      'biolink:Entity',
      'biolink:NamedThing',
      'biolink:ThingWithTaxon'],
     'attributes': [{'original_attribute_name': 'iri',
       'value': 'http://purl.obolibrary.o